# MCP server demo
This notebook connects to the MCP server defined in this repository and
registers it as a tool in an OpenAI Chat Completions request so the model can call the server directly.

In [ ]:
import os, json, urllib.request

SERVER_URL = 'http://localhost:3333/'
API_KEY = os.environ.get('MCP_API_KEY')
if not API_KEY:
    raise RuntimeError('Set MCP_API_KEY environment variable')

def mcp_request(method, params=None, id=1):
    payload = json.dumps({"jsonrpc": "2.0", "id": id, "method": method, "params": params or {}}).encode()
    headers = {'Content-Type': 'application/json', 'x-api-key': API_KEY}
    req = urllib.request.Request(SERVER_URL, data=payload, headers=headers)
    with urllib.request.urlopen(req) as resp:
        return json.load(resp)

mcp_request('list_resources')


In [ ]:
api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    raise RuntimeError('Set OPENAI_API_KEY environment variable')

openai_payload = json.dumps({
    'model': 'gpt-4o-mini',
    'messages': [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': 'Fetch a cat fact using the MCP server and summarize it.'}
    ],
    'tools': [
        {
            'type': 'mcp',
            'server_url': SERVER_URL,
            'api_key': API_KEY
        }
    ]
}).encode()

req = urllib.request.Request(
    'https://api.openai.com/v1/chat/completions',
    data=openai_payload,
    headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {api_key}'}
)
with urllib.request.urlopen(req) as resp:
    reply = json.load(resp)

reply['choices'][0]['message']['content']
